In [16]:
import os
import librosa
import numpy as np
import joblib
from music21 import note, stream


In [17]:
model = joblib.load("rf_13_classifier.pkl")


In [18]:
def create_audio_file(files):
    combined_audio = []
    sr = 16000
    silence = np.zeros(int(0.2 * sr))

    for i, file_name in enumerate(files):
        path = os.path.join("nsynth-train/audio", file_name)
        y, _ = librosa.load(path, sr=sr)
        combined_audio.append(y)

        if i < len(files) - 1:
            combined_audio.append(silence)

    return np.concatenate(combined_audio), sr


In [19]:
def segment_audio_pyin(y, sr):
    f0, voiced_flag, _ = librosa.pyin(
        y,
        fmin=librosa.note_to_hz("A2"),
        fmax=librosa.note_to_hz("A5")
    )

    voiced = np.asarray(voiced_flag, dtype=bool)
    frame_samples = librosa.frames_to_samples(np.arange(len(voiced)))

    energy = librosa.feature.rms(y=y)[0]

    segments = []
    start = None

    for i in range(len(voiced)):

        is_voiced = voiced[i]
        low_energy = energy[i] < 0.01

        if is_voiced and start is None:
            start = frame_samples[i]

        elif (not is_voiced or low_energy) and start is not None:
            end = frame_samples[i]
            duration = (end - start) / sr
            if duration > 0.3:
                segments.append((int(start), int(end)))
            start = None

    if start is not None:
        duration = (len(y) - start) / sr
        if duration > 0.3:
            segments.append((int(start), len(y)))

    return segments


def merge_close_segments(segments, sr, gap_thresh=0.15):

    if len(segments) == 0:
        return []

    merged = []
    prev_start, prev_end = segments[0]

    for start, end in segments[1:]:

        gap = (start - prev_end) / sr

        if gap < gap_thresh:
            prev_end = end
        else:
            merged.append((prev_start, prev_end))
            prev_start, prev_end = start, end

    merged.append((prev_start, prev_end))
    return merged


In [20]:
def extract_segment_features(segment_audio, sr):
    mfcc = np.mean(librosa.feature.mfcc(y=segment_audio, sr=sr, n_mfcc=13).T, axis=0)
    chroma = np.mean(librosa.feature.chroma_stft(y=segment_audio, sr=sr).T, axis=0)
    centroid = np.mean(librosa.feature.spectral_centroid(y=segment_audio, sr=sr))
    zcr = np.mean(librosa.feature.zero_crossing_rate(segment_audio))

    return np.hstack((mfcc, chroma, centroid, zcr))


In [21]:
def predict_segment_pitch(segment_audio, sr):
    features = extract_segment_features(segment_audio, sr)
    features = features.reshape(1, -1)
    pitch = model.predict(features)[0]

    return int(pitch)


In [22]:
def extract_beats(y, sr):
    tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr)
    beat_times = librosa.frames_to_time(beat_frames, sr=sr)

    return tempo, beat_times


def map_segments_to_beats(segments, beat_times, sr, tempo):
    beat_durations = []
    seconds_per_beat = 60 / float(np.asarray(tempo).item())

    for start, end in segments:
        start_time = start / sr
        end_time = end / sr

        if len(beat_times) > 0:
            start_idx = np.argmin(np.abs(beat_times - start_time))
            end_idx = np.argmin(np.abs(beat_times - end_time))
            start_time = beat_times[start_idx]
            end_time = beat_times[end_idx]

        beats = max((end_time - start_time) / seconds_per_beat, 0.25)
        beat_durations.append(beats)

    return beat_durations


def quantize_beats(d):
    allowed = [0.25, 0.5, 1, 2, 4]
    return min(allowed, key=lambda x: abs(x - d))


def build_notes(y, sr, segments, beat_durations):
    notes = []
    durations = []

    for i, (start, end) in enumerate(segments):
        segment_audio = y[start:end]
        if len(segment_audio) == 0:
            continue

        if len(segment_audio) < 2000:
            continue

        if np.mean(np.abs(segment_audio)) < 0.01:
            continue

        pitch = predict_segment_pitch(segment_audio, sr)
        beat_duration = beat_durations[i]

        notes.append(pitch)
        durations.append(beat_duration)

    return notes, durations


In [23]:
def create_seq_sheet(notes, durations, output_name="output"):
    score = stream.Stream()

    for pitch, duration in zip(notes, durations):
        n = note.Note()
        n.pitch.midi = int(pitch)
        n.quarterLength = quantize_beats(duration)
        score.append(n)

    output_dir = os.path.dirname(output_name)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    score.write("musicxml", output_name + ".xml")
    print("Sheet music created at", output_name + ".xml")


In [24]:
def run_pipeline(files, output_name):
    y, sr = create_audio_file(files)
    segments = segment_audio_pyin(y, sr)
    segments = merge_close_segments(segments, sr)
    tempo, beat_times = extract_beats(y, sr)
    beat_durations = map_segments_to_beats(segments, beat_times, sr, tempo)
    notes, durations = build_notes(y, sr, segments, beat_durations)

    create_seq_sheet(notes, durations, output_name)

    print("tempo:", tempo)
    print("detected segments:", segments)
    print("predicted notes:", notes)
    print("beat durations:", beat_durations)
    print("num segments:", len(segments))
    print("num notes:", len(notes))
    print("segment durations:", [(end-start)/sr for start,end in segments])
    print("total duration vs audio length:", sum(durations) * (60 / float(np.asarray(tempo).item())), len(y) / sr)

    return notes, durations, segments


In [26]:
#Example
files = [
    "keyboard_acoustic_000-044-127.wav",
    "keyboard_acoustic_003-048-127.wav",
    "keyboard_acoustic_007-050-075.wav",
    "keyboard_acoustic_011-053-127.wav",
    "keyboard_acoustic_018-045-127.wav"
]
notes, durations, segments = run_pipeline(files, "pitch_segmentation/hybrid_output")


Sheet music created at pitch_segmentation/hybrid_output.xml
tempo: [170.45454545]
detected segments: [(0, 49152), (49664, 50176), (50688, 51200), (51712, 52224), (52736, 53248), (53760, 54272), (54784, 55296), (55808, 56320), (66560, 117760), (118272, 118784), (119296, 119808), (120320, 120832), (121344, 121856), (122368, 122880), (133632, 183296), (183808, 184320), (184832, 185344), (185856, 186368), (186880, 187392), (187904, 188416), (188928, 189440), (189952, 190464), (190976, 191488), (192000, 192512), (193024, 193536), (194048, 194560), (195072, 195584), (196096, 196608), (197120, 197632), (198144, 198656), (201216, 211968), (212480, 212992), (213504, 214016), (214528, 215040), (215552, 216064), (216576, 217088), (217600, 218112), (218624, 219136), (219648, 220160), (220672, 221184), (221696, 222208), (222720, 223232), (223744, 224256), (224768, 225280), (225792, 226304), (226816, 227328), (227840, 228352), (228864, 229376), (229888, 230400), (230912, 231424), (231936, 232448), (